# Altair plots quickstart

Shows how to draw charts inside a notebook using [Altair](https://altair-viz.github.io/),
a declarative plotting library that pairs well with `pandas` DataFrames.

We pull a sample of attributes and events via `mwlab` and render two charts:
a breakdown of attribute types and an event-volume timeline.

Fork this notebook to a personal copy before running.

## 1. Imports

`altair` and `pandas` ship in the lab-worker image.

In [ ]:
import altair as alt
import pandas as pd

# The notebook UI sanitises script tags out of cell HTML for safety, so
# Altair's default vega-embed renderer (which needs JavaScript) won't draw.
# Switch to static SVG via vl-convert — supported by CellOutput.vue.
alt.renderers.enable("svg")

## 2. Attribute-type breakdown

Pull a batch of attributes and chart them by `type`.

In [ ]:
attrs = mwlab.search_attributes(size=200)
df = pd.DataFrame(attrs)

type_counts = (
    df.groupby("type").size().reset_index(name="count").sort_values("count", ascending=False)
)

alt.Chart(type_counts).mark_bar().encode(
    x=alt.X("count:Q", title="attributes"),
    y=alt.Y("type:N", sort="-x", title="attribute type"),
    tooltip=["type", "count"],
).properties(width=500, height=300, title="Attributes by type")

## 3. Event volume over time

Group events by their `date` field and plot a daily bar chart.

In [ ]:
events = mwlab.search_events(size=200)
events_df = pd.DataFrame(events)
events_df["@timestamp"] = pd.to_datetime(events_df["@timestamp"], errors="coerce")

daily = (
    events_df.dropna(subset=["@timestamp"])
    .assign(day=lambda d: d["@timestamp"].dt.floor("D"))
    .groupby("day")
    .size()
    .reset_index(name="count")
)

alt.Chart(daily).mark_bar().encode(
    x=alt.X("day:T", title="day"),
    y=alt.Y("count:Q", title="events"),
    tooltip=["day", "count"],
).properties(width=600, height=250, title="Events per day")